## 13장 합성곱 신경망을 사용한 이미지 분류

### 1. 패션 MNIST 데이터 불러오기

In [1]:
# 필요한 라이브러리를 임포트한다.
from tensorflow import keras
from sklearn.model_selection import train_test_split

# 패션 MNIST 데이터셋을 로드한다.
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

# 넘파이 reshape()를 사용해 전체 배열 차원을 그대로 유지하면서 마지막에 차원을 추가한다.
# 훈련데이터를 0~1 사이의 값으로 정규화한다.
train_scaled = train_input.reshape(-1, 28, 28, 1) / 255.0

# 훈련세트와 검증세트를 분리한다.
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42
)

4431872/4422102 [==============================] - 0s 0us/step


### 2. 합성곱 신경망 만들기

2.1 첫번째 합성곱 - 풀링층 추가

In [2]:
# 신경망 객체를 생성한다.
model = keras.Sequential()

In [3]:
# 신경망 모델에 합성곱 층을 추가한다.
# 필터의 개수는 32, 커널의 크기는 (3,3), 활성화점수는 렐루함수, 패딩은 세임패딩
# 입력크기는 (28, 28, 1)으로 지정한다.
model.add(keras.layers.Conv2D(32, kernel_size=3, activation='relu', padding='same',
                              input_shape=(28,28,1)))

In [4]:
# 신경망 모델에 풀링 층을 추가한다.
# 최대 풀링을 사용하며 풀링 크기는 (2, 2)로 지정한다.
model.add(keras.layers.MaxPooling2D(2))

2.2 두번째 합성곱 - 풀링층 추가

In [5]:
# 신경망 모델에 필터의 개수가 64개인 합성곱 층을 추가한다.
# 필터의 개수는 64, 커널의 크기는 (3, 3), 활성화함수는 렐루함수, 패딩은 세임패딩을 지정한다.
model.add(keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu', padding='same'))

# 신경망 모델에 풀링 층을 추가한다.
# 최대 풀링을 사용하며 풀링 크기는 (2, 2)로 지정한다.
model.add(keras.layers.MaxPooling2D(2))

3.3 세번째 완전연결층

In [7]:
# Flatten 층을 추가한다.
# Flatten 층은 3차원 특성맵을 일렬로 펼친다.
model.add(keras.layers.Flatten())

# 학습, 예측, 평가 계산을 위한 밀집층(은닉층)을 추가한다.
#100개의 뉴런을 사용하고 활성화 함수는 렐루함수를 사용한다.
model.add(keras.layers.Dense(100, activation='relu'))

# 과대적합을 막아 성능을 개선하기 위한 드롭아웃층을 추가한다.
model.add(keras.layers.Dropout(0.4))

# 클래스 분류 확률 계산을 위한 밀집층(출력층)을 추가한다.
# 10개의 뉴런을 사용하고 활성화 함수는 소프트맥스 함수를 사용한다.
model.add(keras.layers.Dense(10, activation='softmax'))

In [8]:
# summary()으로 신경망 모델의 정보를 출력한다.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 14, 14, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 14, 14, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 7, 7, 64)         0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 3136)              0         
                                                                 
 dense (Dense)               (None, 100)               3